# Day 094 Project — Risk-Filtered Strategy Dashboard

Compare three strategies — SMA crossover, RSI mean reversion, and MACD — with and without `RiskManager`. Observe the consistent trade-off: lower return, better max drawdown, sometimes higher Sharpe.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def kelly_fraction(win_rate, avg_win, avg_loss):
    if avg_loss <= 0 or win_rate <= 0 or win_rate >= 1:
        return 0.0
    b = avg_win / avg_loss
    return max(0.0, min(1.0, win_rate - (1 - win_rate) / b))
def is_stopped_out(entry_price, current_price, stop_pct=0.05):
    if entry_price <= 0:
        return False
    return current_price <= entry_price * (1.0 - stop_pct)
def apply_stop_loss(signals, prices, stop_pct=0.05):
    result = signals.copy().astype(float)
    entry_price = None
    for i in range(len(result)):
        if result.iloc[i] == 1:
            if entry_price is None:
                entry_price = float(prices.iloc[i])
            elif is_stopped_out(entry_price, float(prices.iloc[i]), stop_pct):
                result.iloc[i] = 0
                entry_price = None
        else:
            entry_price = None
    return result.astype(int)
def market_drawdown(prices):
    peak = prices.cummax()
    return (prices - peak) / peak

def apply_drawdown_limit(signals, prices, limit=-0.20):
    dd = market_drawdown(prices)
    result = signals.copy().astype(int)
    result[dd < limit] = 0
    return result
class RiskManager:
    def __init__(self, stop_pct=0.05, drawdown_limit=-0.20):
        self.stop_pct = stop_pct
        self.drawdown_limit = drawdown_limit

    def filter(self, signals, prices):
        s = apply_stop_loss(signals, prices, self.stop_pct)
        return apply_drawdown_limit(s, prices, self.drawdown_limit)

    def summary(self, original, filtered):
        n_orig = int((original == 1).sum())
        n_kept = int((filtered == 1).sum())
        return {
            "total_long_bars": n_orig,
            "kept_long_bars":  n_kept,
            "filtered_bars":   n_orig - n_kept,
            "filter_rate":     float((n_orig - n_kept) / max(n_orig, 1)),
        }
def _compute_returns(df):  return df["Close"].pct_change()
def _compute_equity(r):    return (1 + r.fillna(0)).cumprod()
def _max_dd(eq):
    peak = eq.cummax(); return float(((eq - peak) / peak).min())
def _sharpe(r):
    c = r.dropna()
    if len(c) == 0 or c.std() == 0: return 0.0
    return float(c.mean() / c.std() * (252 ** 0.5))
def run_backtest(df, signals, label=""):
    mr  = _compute_returns(df)
    pos = signals.shift(1).fillna(0)
    sr  = pos * mr; eq = _compute_equity(sr)
    c   = sr.dropna(); n = len(c); tr = float(eq.iloc[-1] - 1.0)
    base = 1.0 + tr
    ar  = float(base ** (252.0 / max(n, 1)) - 1) if base > 0 else -1.0
    return {
        "label":             label,
        "total_return":      tr,
        "annualized_return": ar,
        "sharpe_ratio":      _sharpe(sr),
        "max_drawdown":      _max_dd(eq),
        "win_rate":          float((c > 0).sum() / max(n, 1)),
        "n_trades":          int((pos.diff().fillna(0) != 0).sum()),
        "equity":            eq,
    }
def _sma_cross(df, fast=20, slow=50):
    c = df["Close"]
    return (c.rolling(fast).mean() > c.rolling(slow).mean()).fillna(False).astype(int)

def _rsi_mr(df, window=14, oversold=35, overbought=65):
    import warnings
    d = df["Close"].diff()
    g = d.clip(lower=0).rolling(window).mean()
    l = (-d.clip(upper=0)).rolling(window).mean()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    rsi_s = 100 - (100 / (1 + rs))
    sig = pd.Series(float("nan"), index=df.index)
    sig[rsi_s < oversold]   = 1.0
    sig[rsi_s > overbought] = 0.0
    return sig.ffill().fillna(0).astype(int)

def _macd_cross(df, fast=12, slow=26, signal=9):
    c = df["Close"]
    ml = c.ewm(span=fast, adjust=False).mean() - c.ewm(span=slow, adjust=False).mean()
    sl = ml.ewm(span=signal, adjust=False).mean()
    return (ml > sl).astype(int)


In [ ]:
df = _synthetic(n=252)
rm = RiskManager(stop_pct=0.05, drawdown_limit=-0.20)

strategies = [
    ("SMA-cross",    _sma_cross(df)),
    ("RSI-MR",       _rsi_mr(df)),
    ("MACD",         _macd_cross(df)),
]

results = []
for label, sig in strategies:
    raw      = run_backtest(df, sig, label)
    filtered = run_backtest(df, rm.filter(sig, df["Close"]), label + "+Risk")
    info     = rm.summary(sig, rm.filter(sig, df["Close"]))
    results.append((label, raw, filtered, info))


In [ ]:
print(f"{'Strategy':<20} {'Raw Return':>11} {'Risk Return':>11} "
      f"{'Raw MaxDD':>10} {'Risk MaxDD':>10} {'Filter%':>8}")
print("-" * 74)
for label, raw, filt, info in results:
    print(f"{label:<20} "
          f"{raw['total_return']:>11.2%} {filt['total_return']:>11.2%} "
          f"{raw['max_drawdown']:>10.2%} {filt['max_drawdown']:>10.2%} "
          f"{info['filter_rate']:>8.2%}")
